# Iris Classification Pipeline (Companion Notebook)

This notebook reproduces the six main stages implemented in the `iris_project/` Mage project:
1. Load the Iris dataset
2. Validate the data
3. Create a stratified 80:20 train/test split
4. Train a Decision Tree classifier
5. Save actual and predicted labels
6. Evaluate the model

The notebook is a companion, reproducible view of the Mage workflow; it does not replace the Mage project files.


In [1]:
# 1. Import libraries and load the Iris dataset
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score
)

iris = load_iris(as_frame=True)
df = iris.frame
df.head()


 sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  target
               5.1               3.5                1.4               0.2       0
               4.9               3.0                1.4               0.2       0
               4.7               3.2                1.3               0.2       0
               4.6               3.1                1.5               0.2       0
               5.0               3.6                1.4               0.2       0

In [2]:
# 2. Validate the dataset
assert df.isnull().sum().sum() == 0, "Missing values found"
assert df["target"].nunique() == 3, "Expected 3 classes"

print("Rows:", len(df))
print("Missing values:", int(df.isnull().sum().sum()))
print("Number of classes:", df["target"].nunique())
print("Validation passed")


Rows: 150\nMissing values: 0\nNumber of classes: 3\nValidation passed\n

In [3]:
# 3. Prepare the data using a stratified 80:20 split
train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=df["target"]
)

prepared = df.copy()
prepared["split"] = "train"
prepared.loc[test_idx, "split"] = "test"

print("Training rows:", int((prepared["split"] == "train").sum()))
print("Testing rows:", int((prepared["split"] == "test").sum()))
prepared.head()


Training rows: 120\nTesting rows: 30\n

 sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  target split
               5.1               3.5                1.4               0.2       0 train
               4.9               3.0                1.4               0.2       0 train
               4.7               3.2                1.3               0.2       0 train
               4.6               3.1                1.5               0.2       0 train
               5.0               3.6                1.4               0.2       0 train

In [4]:
# 4. Train and test the Decision Tree classifier
max_depth = 3

train = prepared[prepared["split"] == "train"]
test = prepared[prepared["split"] == "test"]

X_train = train.drop(columns=["target", "split"])
y_train = train["target"]
X_test = test.drop(columns=["target", "split"])
y_test = test["target"]

model = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print("Max depth:", max_depth)
print(f"Test accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
assert accuracy >= 0.80, "Model accuracy is below the required threshold"


Max depth: 3\nTest accuracy: 0.9667 (96.67%)\n

In [5]:
# 5. Save the actual and predicted labels
results = pd.DataFrame({
    "actual": y_test.tolist(),
    "predicted": predictions.tolist()
})

results.to_csv("iris_predictions.csv", index=False)
print("Predictions saved to iris_predictions.csv")
print("Number of predictions:", len(results))
results


Predictions saved to iris_predictions.csv\nNumber of predictions: 30\n

 actual  predicted
      0          0
      0          0
      0          0
      0          0
      0          0
      0          0
      0          0
      0          0
      0          0
      0          0
      1          1
      1          1
      1          1
      1          1
      1          1
      1          1
      1          1
      1          2
      1          1
      1          1
      2          2
      2          2
      2          2
      2          2
      2          2
      2          2
      2          2
      2          2
      2          2
      2          2

In [6]:
# 6. Evaluate the model
cm = confusion_matrix(y_test, predictions)

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Score": [
        accuracy_score(y_test, predictions),
        precision_score(y_test, predictions, average="weighted", zero_division=0),
        recall_score(y_test, predictions, average="weighted", zero_division=0),
        f1_score(y_test, predictions, average="weighted", zero_division=0)
    ]
})

print("Confusion Matrix:")
print(cm)
print("\nEvaluation Metrics:")
print(metrics.to_string(index=False))


Confusion Matrix:\n[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]\n\nEvaluation Metrics:\n   Metric    Score
 Accuracy 0.966667
Precision 0.969697
   Recall 0.966667
 F1-score 0.966583\n

## Conclusion

The pipeline loaded and validated 150 Iris records, created a stratified 80:20 split (120 training rows and 30 testing rows), and trained a Decision Tree classifier with `max_depth=3`. The test accuracy was **96.67%**, which is above the 80% threshold. The actual and predicted labels were saved in `iris_predictions.csv`.

The Mage project in `iris_project/` contains the corresponding six executed blocks. This notebook provides a readable, reproducible notebook version of the same workflow.
